# Signal-with-mediation computations

Verifies all values in Sections 4.4 (plain SHAP), 4.5 (Causal SHAP), and 4.6 (desiderata table)
of `main.tex`, and computes two PCI columns: with and without witnesses.

**Model:** $X \to M \to Y$
- $X \sim \mathcal{N}(0.5,\; 0.25)$
- $M = X + \varepsilon_M$, $\varepsilon_M \sim \mathcal{N}(0, 0.1)$
- $Y = M + \varepsilon_Y$, $\varepsilon_Y \sim \mathcal{N}(0, 0.1)$

**Factual instance:** $X^\star = M^\star = Y^\star = 1$, so factual noise $\varepsilon_M^\star = 0$, $\varepsilon_Y^\star = 0$.

**PCI setup (both variants):**
- Suspect $A$, target $B$, alternative $A' \sim P(A)$.
- $ci(y^s, y^n, y^\star) = |y^n - y^\star| - |y^s - y^\star|$.  Since $y^s = B^\star = 1$ always, this reduces to $|y^n - 1|$.
- **Without witnesses ($\mathbf{W}=\emptyset$):** causal propagation from $A'$ to $B$ runs freely through the SCM.
- **With witnesses ($\mathbf{W}=\{W\}$, third variable):** $W$ is fixed at $w^\star=1$ via do-intervention.
- Partial abduction: noise terms for $B$ sampled fresh in necessity world.

In [ ]:
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(42)
N = 2_000_000

mu_X, var_X = 0.5, 0.25
var_eM = 0.1
var_eY = 0.1
var_M  = var_X + var_eM   # 0.35
var_Y  = var_M + var_eY   # 0.45
cov_XM = var_X            # 0.25
cov_XY = var_X            # 0.25
cov_MY = var_M            # 0.35
mu_M = mu_X; mu_Y = mu_X
x_star, m_star, y_star = 1.0, 1.0, 1.0

print(f"Var(X)={var_X}, Var(M)={var_M}, Var(Y)={var_Y}")
print(f"Cov(X,M)={cov_XM}, Cov(X,Y)={cov_XY}, Cov(M,Y)={cov_MY}")

## 1. Optimal predictors

In [ ]:
def f_Y(x, m): return m

Sigma_XY = np.array([[var_X, cov_XY],[cov_XY, var_Y]])
coeff_fM = np.array([cov_XM, cov_MY]) @ np.linalg.inv(Sigma_XY)
def f_M(x, y): return mu_M + coeff_fM[0]*(x - mu_X) + coeff_fM[1]*(y - mu_Y)

Sigma_MY = np.array([[var_M, cov_MY],[cov_MY, var_Y]])
coeff_fX = np.array([cov_XM, cov_XY]) @ np.linalg.inv(Sigma_MY)
def f_X(m, y): return mu_X + coeff_fX[0]*(m - mu_M) + coeff_fX[1]*(y - mu_Y)

print(f"f_M(X,Y) = 0.5 + {coeff_fM[0]:.4f}*(X-0.5) + {coeff_fM[1]:.4f}*(Y-0.5)  [expect 0.5X + 0.5Y]")
print(f"f_X(M,Y) = 0.5 + {coeff_fX[0]:.4f}*(M-0.5) + {coeff_fX[1]:.6f}*(Y-0.5)  [expect (5/7)*(M-0.5)]")
print(f"f_Y(1,1)={f_Y(1,1):.4f}  f_M(1,1)={f_M(1,1):.4f}  f_X(1,1)={f_X(1,1):.4f}  [1, 1, {5/14+0.5:.4f}]")

## 2. Plain SHAP coalition values and Shapley assignments

In [ ]:
def E_cond(tmean, cov_to, var_o, oval, omean):
    return tmean + cov_to / var_o * (oval - omean)

def shapley2(vA, vB, vAB, v0):
    return 0.5*(vA-v0)+0.5*(vAB-vB), 0.5*(vB-v0)+0.5*(vAB-vA)

EY_X1 = E_cond(mu_Y, cov_XY, var_X, x_star, mu_X)
EX_Y1 = E_cond(mu_X, cov_XY, var_Y, y_star, mu_Y)
EM_Y1 = E_cond(mu_M, cov_MY, var_Y, y_star, mu_Y)

vY_e, vY_X, vY_M, vY_XM = mu_M, E_cond(mu_M, cov_XM, var_X, x_star, mu_X), m_star, 1.0
vM_e = f_M(mu_X, mu_Y)
vM_X = 0.5*x_star + 0.5*EY_X1
vM_Y = 0.5*EX_Y1 + 0.5*y_star
vM_XY = f_M(x_star, y_star)
vX_e = f_X(mu_M, mu_Y)
vX_M = f_X(m_star, mu_Y)
vX_Y = f_X(EM_Y1, y_star)
vX_MY = f_X(m_star, y_star)

phi_X_Y_plain, phi_M_Y_plain = shapley2(vY_X, vY_M, vY_XM, vY_e)
phi_X_M_plain, phi_Y_M_plain = shapley2(vM_X, vM_Y, vM_XY, vM_e)
phi_M_X_plain, phi_Y_X_plain = shapley2(vX_M, vX_Y, vX_MY, vX_e)

print("Plain SHAP:")
print(f"  phi_X^Y={phi_X_Y_plain:.4f} [0.250]  phi_M^Y={phi_M_Y_plain:.4f} [0.250]")
print(f"  phi_X^M={phi_X_M_plain:.4f} [0.306]  phi_Y^M={phi_Y_M_plain:.4f} [0.194]")
print(f"  phi_M^X={phi_M_X_plain:.4f} [0.219]  phi_Y^X={phi_Y_X_plain:.4f} [0.139]")

## 3. Causal SHAP coalition values and Shapley assignments

In [ ]:
vcY_e, vcY_X, vcY_M, vcY_XM = mu_M, 1.0, 1.0, 1.0
vcM_e = f_M(mu_X, mu_Y)
vcM_X = 0.5*x_star + 0.5*1.0   # E[Y|do(X=1)] = 1
vcM_Y = 0.5*mu_X + 0.5*y_star  # E[X|do(Y=1)] = mu_X (breaks M->Y)
vcM_XY = f_M(x_star, y_star)
vcX_e = f_X(mu_M, mu_Y)
vcX_M = f_X(m_star, 1.0)        # do(M=1); Y coeff~0
vcX_Y = f_X(mu_M, y_star)       # do(Y=1) breaks M->Y; E[M|do(Y=1)]=mu_M
vcX_MY = f_X(m_star, y_star)

phi_X_Y_c, phi_M_Y_c = shapley2(vcY_X, vcY_M, vcY_XM, vcY_e)
phi_X_M_c, phi_Y_M_c = shapley2(vcM_X, vcM_Y, vcM_XY, vcM_e)
phi_M_X_c, phi_Y_X_c = shapley2(vcX_M, vcX_Y, vcX_MY, vcX_e)

print("Causal SHAP:")
print(f"  phi_X^Y={phi_X_Y_c:.4f} [0.250]  phi_M^Y={phi_M_Y_c:.4f} [0.250]")
print(f"  phi_X^M={phi_X_M_c:.4f} [0.375]  phi_Y^M={phi_Y_M_c:.4f} [0.125]")
print(f"  phi_M^X={phi_M_X_c:.4f} [0.357]  phi_Y^X={phi_Y_X_c:.4f} [0.000]")

## 4. PCI — helper and samples

In [ ]:
def E_abs_N(mu, sigma2):
    """E[|Z|] for Z ~ N(mu, sigma2) using the folded-normal formula."""
    sigma = np.sqrt(sigma2)
    c = mu / sigma
    return mu * (2*norm.cdf(c) - 1) + 2*sigma*norm.pdf(c)

X_alt  = rng.normal(mu_X, np.sqrt(var_X), N)
M_alt  = rng.normal(mu_M, np.sqrt(var_M), N)
eM_s   = rng.normal(0, np.sqrt(var_eM), N)
eY_s   = rng.normal(0, np.sqrt(var_eY), N)

## 5. PCI without witnesses ($\mathbf{W} = \emptyset$)

Alternative $A'$ propagates through the full SCM to the target $B$; no variable is pinned.

In [ ]:
# DXY (W=empty): do(X=X'). Y = X' + eM + eY.  y^n - 1 ~ N(-0.5, 0.45)
pci_DXY_nw = E_abs_N(mu_X - y_star, var_X + var_eM + var_eY)
mc_DXY_nw  = np.mean(np.abs(X_alt + eM_s + eY_s - y_star))

# DMY (W=empty): do(M=M'). Y = M' + eY.  y^n - 1 ~ N(-0.5, 0.45)
pci_DMY_nw = E_abs_N(mu_M - y_star, var_M + var_eY)
mc_DMY_nw  = np.mean(np.abs(M_alt + eY_s - y_star))

# DXM (W=empty): do(X=X'). M = X' + eM.  y^n - 1 ~ N(-0.5, 0.35)
pci_DXM_nw = E_abs_N(mu_X - m_star, var_X + var_eM)
mc_DXM_nw  = np.mean(np.abs(X_alt + eM_s - m_star))

# DYX, DYM, DMX (W=empty): suspect does not cause target in X->M->Y; ci=0 exactly.
pci_DYX_nw = pci_DYM_nw = pci_DMX_nw = 0.0

print("PCI without witnesses (closed-form | MC):")
print(f"  DXY: {pci_DXY_nw:.4f} | {mc_DXY_nw:.4f}  — y^n ~ N(-0.5, 0.45)")
print(f"  DMY: {pci_DMY_nw:.4f} | {mc_DMY_nw:.4f}  — y^n ~ N(-0.5, 0.45)")
print(f"  DXM: {pci_DXM_nw:.4f} | {mc_DXM_nw:.4f}  — y^n ~ N(-0.5, 0.35)")
print(f"  DYX: {pci_DYX_nw:.4f}  DYM: {pci_DYM_nw:.4f}  DMX: {pci_DMX_nw:.4f}")
print()
print("KEY: DXY = DMY = 0.677 exactly without witnesses.")
print(f"  Var(X)+Var(eM)+Var(eY) = {var_X+var_eM+var_eY:.2f} = Var(M)+Var(eY) = {var_M+var_eY:.2f}")
print("  The path-variance from X' to Y equals the path-variance from M' to Y.")
print("  => DMXY FAILS without witnesses (same failure as plain/causal SHAP).")

## 6. PCI with witnesses ($\mathbf{W} = $ third variable, $w^\star = 1$)

The third variable (neither $A$ nor $B$) is fixed at its factual value by do-intervention.

In [ ]:
# DXY (W=M=1): do(X=X'), do(M=1). Y = 1 + eY.  Witness blocks X->M->Y.
pci_DXY_w = E_abs_N(0.0, var_eY)
mc_DXY_w  = np.mean(np.abs(1.0 + eY_s - y_star))

# DMY (W=X=1): do(M=M'), do(X=1). Y = M' + eY.  y^n - 1 ~ N(-0.5, 0.45)
pci_DMY_w = E_abs_N(mu_M - y_star, var_M + var_eY)
mc_DMY_w  = np.mean(np.abs(M_alt + eY_s - y_star))

# DXM (W=Y=1): do(X=X'), do(Y=1). M = X' + eM.  do(Y) does not affect M.
pci_DXM_w = E_abs_N(mu_X - m_star, var_X + var_eM)
mc_DXM_w  = np.mean(np.abs(X_alt + eM_s - m_star))

# DYX, DYM, DMX: suspect does not cause target; ci=0 exactly.
pci_DYX_w = pci_DYM_w = pci_DMX_w = 0.0

print("PCI with witnesses (closed-form | MC):")
print(f"  DXY (W=M): {pci_DXY_w:.4f} | {mc_DXY_w:.4f}  — witness blocks X->M->Y; residual = noise eY")
print(f"  DMY (W=X): {pci_DMY_w:.4f} | {mc_DMY_w:.4f}  — M directly causes Y")
print(f"  DXM (W=Y): {pci_DXM_w:.4f} | {mc_DXM_w:.4f}  — do(Y) does not affect M")
print(f"  DYX: {pci_DYX_w:.4f}  DYM: {pci_DYM_w:.4f}  DMX: {pci_DMX_w:.4f}")
print()
print(f"DMXY with witnesses: PCI(M->Y|W=X)={pci_DMY_w:.3f} > PCI(X->Y|W=M)={pci_DXY_w:.3f}  =>  satisfied")
print("  (Different witnesses; the witness severs X's indirect path, exposing the direct/indirect gap.)")

## 7. Full desiderata table

In [ ]:
T = "\u2713"; X = "\u00d7"

def mark(v, ok_fn):
    return f"{v:.3f} {T if ok_fn(v) else X}"

def gt0(v): return v > 1e-9
def eq0(v): return abs(v) < 1e-9

rows = [
    ("DXY",  "phi_X^Y > 0",
     phi_X_Y_plain, phi_X_Y_c, pci_DXY_nw, pci_DXY_w, gt0),
    ("DMY",  "phi_M^Y > 0",
     phi_M_Y_plain, phi_M_Y_c, pci_DMY_nw, pci_DMY_w, gt0),
    ("DXM",  "phi_X^M > 0",
     phi_X_M_plain, phi_X_M_c, pci_DXM_nw, pci_DXM_w, gt0),
    ("DYX",  "phi_Y^X = 0",
     phi_Y_X_plain, phi_Y_X_c, pci_DYX_nw, pci_DYX_w, eq0),
    ("DYM",  "phi_Y^M = 0",
     phi_Y_M_plain, phi_Y_M_c, pci_DYM_nw, pci_DYM_w, eq0),
    ("DMX",  "phi_M^X = 0",
     phi_M_X_plain, phi_M_X_c, pci_DMX_nw, pci_DMX_w, eq0),
]

print(f"{'Desid':<5}  {'Condition':<18}  {'Plain':>9}  {'Causal':>9}  {'PCI W=∅':>9}  {'PCI W=3rd':>10}")
print("-" * 72)
for (d, cond, plain, causal, pci_nw, pci_w, ok) in rows:
    print(f"{d:<5}  {cond:<18}  {mark(plain,ok):>9}  {mark(causal,ok):>9}  "
          f"{mark(pci_nw,ok):>9}  {mark(pci_w,ok):>10}")

print()
print("DMXY  phi_M^Y > phi_X^Y")
print(f"  Plain:      {phi_M_Y_plain:.3f} = {phi_X_Y_plain:.3f}  {X}")
print(f"  Causal:     {phi_M_Y_c:.3f} = {phi_X_Y_c:.3f}  {X}")
print(f"  PCI W=empty: {pci_DMY_nw:.3f} = {pci_DXY_nw:.3f}  {X}  (path-variances cancel)")
print(f"  PCI W=3rd:  {pci_DMY_w:.3f} > {pci_DXY_w:.3f}  {T}  (different witnesses)")
print()
print(f"D-ind  sum phi_i^X vs X-E[X]=0.5")
print(f"  Plain: {phi_M_X_plain+phi_Y_X_plain:.3f}  {X}   Causal: {phi_M_X_c+phi_Y_X_c:.3f}  {X}   PCI: n/a")

## 8. Key findings

**Without witnesses (W=∅):** PCI satisfies all 6 single-variable desiderata, but **DMXY fails** —
the same failure as both SHAP variants. The reason is structural: $X' \to M \to Y$ propagates
with total variance $\mathrm{Var}(X)+\mathrm{Var}(\varepsilon_M)+\mathrm{Var}(\varepsilon_Y) = 0.45$,
identical to $\mathrm{Var}(M)+\mathrm{Var}(\varepsilon_Y) = 0.45$ from $M' \to Y$.
Without a mechanism to block the indirect path, PCI cannot distinguish direct from indirect causation.

**With witnesses (W = third variable):** PCI satisfies all 6 single-variable desiderata **and** DMXY.
Pinning $M = m^\star = 1$ severs $X$'s indirect path, so $\mathrm{PCI}(X \to Y \mid M) \approx 0.252$
reflects only outcome noise $\varepsilon_Y$, while $\mathrm{PCI}(M \to Y \mid X) \approx 0.677$
captures $M$'s direct causal role. The witness is precisely the ingredient that distinguishes
direct from indirect effects.

**Caveat on DXY with witnesses (0.252 ✓):** The positive value is driven by noise, not $X$'s causal
signal. The witness over-blocks. Whether this counts as satisfying DXY depends on whether
one reads $\phi_X^Y > 0$ as requiring the attribution to track the *causal signal* or just
to be positive. Numerically it satisfies the stated condition.

**D-ind:** PCI has no local-accuracy axiom; individual PCI values are not designed to sum
to $B^\star - \mathbb{E}[B]$. The desideratum is not well-posed for PCI.